# Coffee17 W0-AT — W0 Spatial Attention Transfer

Eksperimen ini menguji transfer training-only dari **W0 -> R0** pada level attention map spasial ter-normalisasi di MobileNetV3-Large stage 3.

Student/deployment tetap raw RGB. W0 hanya dipakai saat training. Tidak ada auxiliary classifier, output KD, final-vector MVFD, full-tensor MSE, HBP, atau extra inference branch.

**Sebelum Run All:** cukup Add Input **Coffee Green Bean with 17 Defects Original**, aktifkan GPU dan Internet. Jangan attach checkpoint eksperimen lama.

Notebook merekonstruksi sendiri Coffee17 clean population + 5 development folds dari original dataset dan memverifikasi validation identity terhadap frozen matched-R0 reference. **Outer test tidak disentuh.**

Jika run terputus, simpan output notebook ini sebagai Version lalu Add Input output **notebook W0-AT ini sendiri** untuk resume.


In [ ]:
# Coffee17 W0-AT — latest Kaggle experiment pattern
SCIENTIFIC_CODE_COMMIT = "50b6616225b3ba386d00c5c71920a24311bf5212"

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-w0-at-code"
PROJECT = WORK / "coffee17-w0-at-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source, target = Path(source), Path(target)
    for item in sorted(source.rglob("*")):
        if not item.is_file():
            continue
        rel = item.relative_to(source)
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file():
            if sha256_file(item) != sha256_file(dst):
                raise RuntimeError(f"Kaggle input conflict: {rel}")
        else:
            shutil.copy2(item, dst)

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Optional resume from saved output of THIS notebook only.
for prior in sorted(
    p for p in INPUT.rglob("coffee17-w0-at-project")
    if p.is_dir()
):
    print("MERGE PRIOR W0-AT OUTPUT:", prior)
    merge_tree_exact(prior, PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Exact scientific code checkout.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.engine.shared_multiview import validation_identity_label_sha256

REFERENCE = REPO / "configs/shared_multiview/sbn_reference_v1.json"
CONFIG = REPO / "configs/w0_at/W0_AT.yaml"
OUT = PROJECT / "experiments/coffee17-w0-at-v1"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 folds from original dataset only.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")
by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus memuat Coffee17 original valid saja "
        f"(expected 979/17, observed {raw_count}/{len(by_class)})."
    )

ARCHIVE = WORK / "coffee17_w0_at_original.zip"
PROV_LOCAL = WORK / "coffee17_w0_at_provenance"
CANONICAL = WORK / "coffee17_w0_at_original_v1"
FOLDS_LOCAL = WORK / "coffee17_w0_at_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV_LOCAL,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

reference = json.loads(REFERENCE.read_text(encoding="utf-8"))
print("MATCHED R0 REFERENCE SHA:", sha256_file(REFERENCE))
print("EXPECTED PRIMARY INIT   :", reference["expected_initial_model_state_sha256"])

# ------------------------------------------------------------
# 3. Five W0-AT folds. No R0 retraining and no outer test.
# ------------------------------------------------------------
print("\n=== 2/4 W0-AT FIVE FOLDS ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_w0_at_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    count, val_sha = validation_identity_label_sha256(DEV)
    expected = reference["folds"][f"fold_{fold}"]
    print("VAL COUNT:", count)
    print("VAL SHA  :", val_sha)
    if count != expected["count"] or val_sha != expected["identity_label_sha256"]:
        raise RuntimeError(f"Fold {fold}: validation authority mismatch.")
    print("VALIDATION REFERENCE: PASS")

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_w0_at",
        "--config", CONFIG,
        "--reference", REFERENCE,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"W0_AT_fold{fold}.log")

    result = json.loads(
        (
            OUT / "W0_AT" / f"fold_{fold}" / "seed42" / "result.json"
        ).read_text(encoding="utf-8")
    )
    d = result["delta_vs_r0_control"]
    print(
        f"FOLD {fold} | "
        f"R0={result['frozen_matched_r0_control']['macro_f1']:.4f} | "
        f"W0-AT={result['metrics']['macro_f1']:.4f} | "
        f"Macro delta={d['macro_f1']:+.4f} | "
        f"Hard delta={d['hard_class_f1']:+.4f} | "
        f"Worst delta={d['worst_class_f1']:+.4f} | "
        f"epoch1 AT/CE={result['epoch1_weighted_at_to_ce_ratio']:.4f} | "
        f"att-cos={result['best_epoch_attention_cosine']:.4f}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Summary + compact analysis package.
# ------------------------------------------------------------
print("\n=== 3/4 SUMMARY ===")
SUMMARY = PROJECT / "analysis" / "w0_at_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_w0_at_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]

print("\n=== KEY RESULT ===")
print("R0 CONTROL Macro-F1:", f"{agg['R0_CONTROL']['macro_f1']['mean']:.4%}")
print("W0-AT Macro-F1     :", f"{agg['W0_AT']['macro_f1']['mean']:.4%}")
print("Delta Macro-F1     :", f"{agg['DELTA']['macro_f1']['mean']:+.4%}")
print("Delta Hard-F1      :", f"{agg['DELTA']['hard_class_f1']['mean']:+.4%}")
print("Delta Worst-F1     :", f"{agg['DELTA']['worst_class_f1']['mean']:+.4%}")
print("Positive Macro folds:", summary["positive_macro_folds"], "/ 5")
print("Positive Hard folds :", summary["positive_hard_folds"], "/ 5")
print("SCREENING GATE      :", summary["screening_gate"]["decision"])
print(
    "Epoch1 weighted AT/CE:",
    f"{agg['EPOCH1_WEIGHTED_AT_TO_CE_RATIO']['mean']:.4f}"
)
print(
    "Best-epoch attention cosine:",
    f"{agg['BEST_EPOCH_ATTENTION_COSINE']['mean']:.4f}"
)

print("\n=== 4/4 BUILD ANALYSIS PACKAGE ===")
PKG = WORK / "w0-at-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(PROJECT / "analysis", PKG / "analysis")
for fold in range(1, 6):
    src = OUT / "W0_AT" / f"fold_{fold}" / "seed42"
    dst = PKG / "W0_AT" / f"fold_{fold}"
    dst.mkdir(parents=True, exist_ok=True)
    for name in (
        "result.json",
        "run_contract.json",
        "history.json",
        "run_config.yaml",
        "loss_scale_failure.json",
    ):
        p = src / name
        if p.is_file():
            shutil.copy2(p, dst / name)
    if (src / "validation").is_dir():
        shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(REFERENCE, PKG / "matched_r0_reference.json")
shutil.copy2(CONFIG, PKG / "W0_AT.yaml")
protocol = REPO / "docs/protocols/W0_AT_V1.md"
if protocol.is_file():
    shutil.copy2(protocol, PKG / "W0_AT_V1.md")

zip_path = Path(
    shutil.make_archive(
        str(WORK / "w0-at-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("PROJECT:", PROJECT)
print("Tidak ada outer-test inference dan tidak membutuhkan checkpoint lama.")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
